# Ανάλυση Αιτήματος Δαπανών

Αυτό το σημειωματάριο παρουσιάζει πώς να δημιουργήσετε πράκτορες που χρησιμοποιούν πρόσθετα για την επεξεργασία εξόδων ταξιδιού από τοπικές εικόνες αποδείξεων, την παραγωγή ενός email αιτήματος δαπανών και την οπτικοποίηση δεδομένων εξόδων χρησιμοποιώντας ένα γράφημα πίτας. Οι πράκτορες επιλέγουν δυναμικά λειτουργίες βάσει του πλαισίου της εργασίας.

Βήματα:
1. Ο πράκτορας OCR επεξεργάζεται την τοπική εικόνα απόδειξης και εξάγει δεδομένα εξόδων ταξιδιού.
2. Ο πράκτορας Email δημιουργεί ένα email αιτήματος δαπανών.

### Παράδειγμα σεναρίου εξόδων ταξιδιού:
Φανταστείτε ότι είστε υπάλληλος που ταξιδεύει για μια επαγγελματική συνάντηση σε άλλη πόλη. Η εταιρεία σας έχει πολιτική να αποζημιώνει όλα τα λογικά έξοδα που σχετίζονται με το ταξίδι. Ακολουθεί ανάλυση πιθανών εξόδων ταξιδιού:
- Μεταφορές:
Αεροπορικά εισιτήρια για ταξίδι με επιστροφή από την πόλη σας στην πόλη προορισμού.
Ταξί ή υπηρεσίες μεταφοράς από και προς το αεροδρόμιο.
Τοπικές μεταφορές στην πόλη προορισμού (όπως δημόσια συγκοινωνία, ενοικιάσεις αυτοκινήτων ή ταξί).

- Διαμονή:
Διαμονή σε ξενοδοχείο για τρεις νύχτες σε μεσαίας κατηγορίας επιχειρηματικό ξενοδοχείο κοντά στον χώρο της συνάντησης.

- Γεύματα:
Ημερήσιο επίδομα γευμάτων για πρωινό, μεσημεριανό και δείπνο, βάσει της πολιτικής ημερήσιας αποζημίωσης της εταιρείας.

- Διάφορα έξοδα:
Τέλη στάθμευσης στο αεροδρόμιο.
Χρεώσεις πρόσβασης στο ίντερνετ στο ξενοδοχείο.
Φιλοδωρήματα ή μικρές χρεώσεις υπηρεσιών.

- Τεκμηρίωση:
Υποβάλλετε όλες τις αποδείξεις (πτήσεις, ταξί, ξενοδοχείο, γεύματα κ.λπ.) και μια συμπληρωμένη αναφορά εξόδων για αποζημίωση.


## Εισαγωγή απαιτούμενων βιβλιοθηκών

Εισάγετε τις απαραίτητες βιβλιοθήκες και μονάδες για το σημειωματάριο.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Ορισμός Μοντέλων Δαπανών

 Δημιουργήστε ένα μοντέλο Pydantic για ατομικές δαπάνες και μια κλάση ExpenseFormatter για τη μετατροπή ενός ερωτήματος χρήστη σε δομημένα δεδομένα δαπανών.

 Κάθε δαπάνη θα αναπαρίσταται στη μορφή:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Ορισμός Εργαλείων - Δημιουργία του Email

Δημιουργήστε μια λειτουργία εργαλείου για τη δημιουργία ενός email υποβολής αίτησης δαπανών.
- Αυτό το εργαλείο χρησιμοποιεί το διακοσμητή `@tool` από το Microsoft Agent Framework.
- Υπολογίζει το συνολικό ποσό των δαπανών και μορφοποιεί τις λεπτομέρειες σε ένα σώμα email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Εργαλείο για την Εξαγωγή Εξόδων Ταξιδιού από Εικόνες Αποδείξεων

Δημιουργήστε μια λειτουργία εργαλείου για την εξαγωγή εξόδων ταξιδίου από εικόνες αποδείξεων.
- Αυτό το εργαλείο χρησιμοποιεί το διακοσμητή `@tool` από το Microsoft Agent Framework.
- Διαβάζει την εικόνα απόδειξης, την κωδικοποιεί σε base64 και επιστρέφει το URI δεδομένων για να αναλυθεί από τον πράκτορα.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Επεξεργασία Εξόδων

Ορίστε τους πράκτορες και συνδέστε τους σε μια διαδοχική ροή εργασίας χρησιμοποιώντας το `WorkflowBuilder`.  
- Ο πράκτορας OCR εξάγει δομημένα δεδομένα εξόδων από την εικόνα της απόδειξης χρησιμοποιώντας το εργαλείο `load_receipt_image`.  
- Ο πράκτορας Email παίρνει τα εξαγόμενα δεδομένα και δημιουργεί ένα επαγγελματικό email αξίωσης εξόδων χρησιμοποιώντας το εργαλείο `generate_expense_email`.  
- Το `WorkflowBuilder` με το `add_edge` δημιουργεί μια διαδοχική αλυσίδα: Πράκτορας OCR → Πράκτορας Email.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Κύρια λειτουργία

Δημιουργήστε τη διαδοχική ροή εργασίας και εκτελέστε την για να επεξεργαστείτε την εικόνα της απόδειξης και να δημιουργήσετε το email αίτησης εξόδων.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:  
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία αυτόματης μετάφρασης AI [Co-op Translator](https://github.com/Azure/co-op-translator). Παρόλο που επιδιώκουμε την ακρίβεια, παρακαλούμε λάβετε υπόψη ότι οι αυτόματες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το αρχικό έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες προτείνεται η επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρανοήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
